# ETF V2.1：V2 基线 + 单一量价交互 A/B

目标：严格复用 ETF V2 的数据、标签、训练和评分链路；每次只改变一个成交额门槛或一个量价交互特征，并导出可由 JoinQuant 回测文件直接加载的 `pkl`。

本版刻意保持简单：

- 标的池：历史时点可交易 ETF，剔除债券、货币、现金管理类 ETF。
- 调仓频率：周频，使用每周最后一个交易日作为 feature date，回测文件在下一周第一个交易日调仓时用 `context.previous_date` 取同口径特征。
- Feature：只用调仓日前历史 OHLCV/money 派生出的动量、波动、回撤、均线、成交额、趋势 R²、趋势分数、池内趋势广度和横截面 rank 特征。
- Label：未来 5 个交易日收益 `future_ret_5d`，以及相对当周 ETF 横截面中位数的 `target_alpha_5d`。
- 模型：LightGBM 回归，无 early stopping；如研究环境缺 LightGBM，则退化为 sklearn 随机森林，bundle 会记录 backend。
- 输出：V2 数据面板、模型 pkl、周度离线 top3 结果、summary、latest targets。

注意：这个 notebook 是 ETF 数据重建和模型导出专用，需要在 JoinQuant 研究环境运行；如果已有 CSV，则可以关闭 `REBUILD_DATA` 直接读缓存。

In [1]:
# =========================
# 0. Config
# =========================
import os
import gc
import pickle
import datetime
import math
import numpy as np
import pandas as pd
from jqdata import *
try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

# 默认依次产出两组独立模型；每组使用 V2 的同一数据与训练链路，输出自动隔离。
LIQUIDITY_TAGS_TO_RUN = ["money2000w", "money5000w"]
LIQUIDITY_TAG = LIQUIDITY_TAGS_TO_RUN[0]
VOLUME_AB_MODE = "baseline"  # baseline / ret5_volume_confirm / trend20_volume_confirm

LIQUIDITY_THRESHOLDS = {"money2000w": 20000000.0, "money5000w": 50000000.0}
if LIQUIDITY_TAG not in LIQUIDITY_THRESHOLDS:
    raise ValueError("invalid LIQUIDITY_TAG: %s" % LIQUIDITY_TAG)
RUN_TAG = LIQUIDITY_TAG + "_" + VOLUME_AB_MODE
MIN_AVG_MONEY_20 = LIQUIDITY_THRESHOLDS[LIQUIDITY_TAG]

if VOLUME_AB_MODE not in ["baseline", "ret5_volume_confirm", "trend20_volume_confirm"]:
    raise ValueError("invalid VOLUME_AB_MODE: %s" % VOLUME_AB_MODE)

OUT_DIR = "etf_ml_v2_1_" + RUN_TAG + "_outputs"
PANEL_CSV = os.path.join(OUT_DIR, "etf_ml_v2_weekly_panel.csv")
SCORE_CSV = os.path.join(OUT_DIR, "etf_ml_v2_score_panel.csv")
SUMMARY_CSV = os.path.join(OUT_DIR, "etf_ml_v2_model_summary.csv")
WEEKLY_CSV = os.path.join(OUT_DIR, "etf_ml_v2_weekly_top3_proxy.csv")
LATEST_TARGETS_CSV = os.path.join(OUT_DIR, "etf_ml_v2_latest_targets.csv")
MODEL_MANIFEST_CSV = os.path.join(OUT_DIR, "etf_ml_v2_model_manifest.csv")

REBUILD_DATA = True
START_DATE = "2021-01-01"
END_DATE = "2026-06-30"
LOOKBACK_DAYS = 60
LABEL_HORIZON_DAYS = 5
MIN_LISTING_DAYS = 180
ETF_CHUNK_SIZE = 120
STOCK_NUM = 3
BENCHMARK = "000985.XSHG"
RANDOM_SEED = 42

# V2: add weighted log-price trend quality features inspired by the manual ETF system.
TREND_WINDOWS = [10, 20, 25, 60]
TREND_WEIGHT_END = 2.0
PRICE_HISTORY_COUNT = max(LOOKBACK_DAYS + 1, max(TREND_WINDOWS) + 1)
POOL_CONTEXT_WINDOW = 25

# 这些关键词会在训练和回测两端同时剔除，避免模型靠债券/货币 ETF 的低波特征混淆目标。
EXCLUDE_NAME_KEYWORDS = [
    "债", "国债", "地债", "政金债", "公司债", "城投", "可转债",
    "货币", "现金", "快线", "快钱", "同业存单",
]

# 多个训练截止日只改变训练时间，不改变 feature/label/model 逻辑。
TRAIN_WINDOWS = [
    ("train20210101_20231231", "2021-01-01", "2023-12-31"),
    ("train20210101_20241231", "2021-01-01", "2024-12-31"),
    ("train20210101_20251231", "2021-01-01", "2025-12-31"),
]

# V2.1 只保留 V2 中表现更直接的 future_ret_5d 回归标签。
TARGET_SPECS = [("ret5d", "future_ret_5d")]

BASE_PRICE_FEATURE_COLS = [
    "ret_1", "ret_5", "ret_10", "ret_20", "ret_60",
    "vol_5", "vol_20", "vol_60",
    "close_to_ma20", "close_to_ma60", "ma5_to_ma20", "ma20_to_ma60",
    "drawdown_20", "drawdown_60",
    "amp_20", "amp_60",
    "money_mean_20", "money_ratio_5_20", "money_ratio_20_60",
    "volume_ratio_5_20", "volume_ratio_20_60",
    "max_ret_20", "min_ret_20",
]
TREND_FEATURE_COLS = []
for w in TREND_WINDOWS:
    TREND_FEATURE_COLS.extend([
        "trend_ann_%s" % w,
        "trend_r2_%s" % w,
        "trend_score_%s" % w,
        "trend_vol_%s" % w,
        "trend_score_vol_adj_%s" % w,
        "trend_simple_ann_%s" % w,
    ])
CONTEXT_FEATURE_COLS = [
    "pool_breadth_%s" % POOL_CONTEXT_WINDOW,
    "pool_median_vol_%s" % POOL_CONTEXT_WINDOW,
]
RAW_FEATURE_COLS = BASE_PRICE_FEATURE_COLS + TREND_FEATURE_COLS
RANK_FEATURE_COLS = ["rank_" + c for c in RAW_FEATURE_COLS]
FEATURE_COLS = RAW_FEATURE_COLS + RANK_FEATURE_COLS + CONTEXT_FEATURE_COLS
VOLUME_AB_FEATURE_COL = {
    "baseline": None,
    "ret5_volume_confirm": "ret_5_x_rank_money_ratio_5_20",
    "trend20_volume_confirm": "trend_score_20_x_rank_money_ratio_5_20",
}[VOLUME_AB_MODE]
if VOLUME_AB_FEATURE_COL is not None:
    FEATURE_COLS = FEATURE_COLS + [VOLUME_AB_FEATURE_COL]

LGB_PARAMS = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.04,
    "num_leaves": 31,
    "max_depth": 5,
    "min_data_in_leaf": 80,
    "feature_fraction": 0.90,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 1.0,
    "min_gain_to_split": 0.0,
    "verbose": -1,
    "seed": RANDOM_SEED,
}
NUM_BOOST_ROUND = 180

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

print("price history count:", PRICE_HISTORY_COUNT)
print("features:", len(FEATURE_COLS), FEATURE_COLS)
print("labels:", TARGET_SPECS)
print("run tag / volume mode:", RUN_TAG, VOLUME_AB_MODE)
print("out dir:", OUT_DIR)

price history count: 61
features: 96 ['ret_1', 'ret_5', 'ret_10', 'ret_20', 'ret_60', 'vol_5', 'vol_20', 'vol_60', 'close_to_ma20', 'close_to_ma60', 'ma5_to_ma20', 'ma20_to_ma60', 'drawdown_20', 'drawdown_60', 'amp_20', 'amp_60', 'money_mean_20', 'money_ratio_5_20', 'money_ratio_20_60', 'volume_ratio_5_20', 'volume_ratio_20_60', 'max_ret_20', 'min_ret_20', 'trend_ann_10', 'trend_r2_10', 'trend_score_10', 'trend_vol_10', 'trend_score_vol_adj_10', 'trend_simple_ann_10', 'trend_ann_20', 'trend_r2_20', 'trend_score_20', 'trend_vol_20', 'trend_score_vol_adj_20', 'trend_simple_ann_20', 'trend_ann_25', 'trend_r2_25', 'trend_score_25', 'trend_vol_25', 'trend_score_vol_adj_25', 'trend_simple_ann_25', 'trend_ann_60', 'trend_r2_60', 'trend_score_60', 'trend_vol_60', 'trend_score_vol_adj_60', 'trend_simple_ann_60', 'rank_ret_1', 'rank_ret_5', 'rank_ret_10', 'rank_ret_20', 'rank_ret_60', 'rank_vol_5', 'rank_vol_20', 'rank_vol_60', 'rank_close_to_ma20', 'rank_close_to_ma60', 'rank_ma5_to_ma20', 'ran

In [2]:
# =========================
# 1. Feature and label definitions
# =========================
feature_doc = pd.DataFrame([
    ("ret_1/5/10/20/60", "过去 N 日收益率，捕捉短中期 ETF 动量/反转"),
    ("vol_5/20/60", "过去 N 日日收益波动率，捕捉风险和拥挤程度"),
    ("close_to_ma20/60", "当前价格相对均线位置，捕捉趋势强弱"),
    ("ma5_to_ma20/ma20_to_ma60", "短中长期均线结构，捕捉趋势斜率"),
    ("drawdown_20/60", "当前价格相对 N 日高点回撤，捕捉趋势损伤"),
    ("amp_20/60", "日内 high/low 振幅均值，捕捉波动和交易噪声"),
    ("money_mean_20", "20 日平均成交额，用于容量和流动性"),
    ("money_ratio/volume_ratio", "短期成交活跃度相对中期变化"),
    ("max_ret_20/min_ret_20", "过去 20 日极端单日收益，捕捉跳涨/跳跌"),
    ("trend_ann/r2/score", "log price 加权线性趋势、趋势拟合 R²、趋势强度乘趋势质量"),
    ("trend_score_vol_adj", "趋势分数除以波动率，惩罚高噪声趋势"),
    ("pool_breadth/median_vol", "同一周 ETF 池内正趋势广度和中位波动，作为市场状态特征"),
    ("rank_*", "同一 feature date ETF 横截面 rank pct，降低量纲和阶段漂移影响"),
], columns=["feature", "meaning"])
label_doc = pd.DataFrame([
    ("future_ret_5d", "feature date 收盘到 5 个交易日后收盘的收益率"),
    ("target_alpha_5d", "future_ret_5d 减去同一 feature date 全 ETF 样本中位数收益"),
], columns=["label", "meaning"])
display(feature_doc)
display(label_doc)

,feature,meaning
0,ret_1/5/10/20/60,过去 N 日收益率，捕捉短中期 ETF 动量/反转
1,vol_5/20/60,过去 N 日日收益波动率，捕捉风险和拥挤程度
2,close_to_ma20/60,当前价格相对均线位置，捕捉趋势强弱
3,ma5_to_ma20/ma20_to_ma60,短中长期均线结构，捕捉趋势斜率
4,drawdown_20/60,当前价格相对 N 日高点回撤，捕捉趋势损伤
5,amp_20/60,日内 high/low 振幅均值，捕捉波动和交易噪声
6,money_mean_20,20 日平均成交额，用于容量和流动性
7,money_ratio/volume_ratio,短期成交活跃度相对中期变化
8,max_ret_20/min_ret_20,过去 20 日极端单日收益，捕捉跳涨/跳跌
9,trend_ann/r2/score,log price 加权线性趋势、趋势拟合 R²、趋势强度乘趋势质量


,label,meaning
0,future_ret_5d,feature date 收盘到 5 个交易日后收盘的收益率
1,target_alpha_5d,future_ret_5d 减去同一 feature date 全 ETF 样本中位数收益


In [3]:
# =========================
# 2. JoinQuant data helpers
# =========================
def to_timestamp(x):
    return pd.Timestamp(x).normalize()


def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def get_weekly_feature_dates(start_date, end_date):
    try:
        days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    except NameError:
        raise RuntimeError("This data rebuild cell must run in JoinQuant research environment, or set REBUILD_DATA=False and provide PANEL_CSV.")
    if len(days) == 0:
        return []
    s = pd.Series(days)
    out = []
    for _, gdf in s.groupby(s.dt.strftime("%Y-%W")):
        out.append(pd.Timestamp(gdf.max()).normalize())
    return out


def should_exclude_name(name):
    text = str(name)
    for kw in EXCLUDE_NAME_KEYWORDS:
        if kw and kw in text:
            return True
    return False


def get_etf_universe_on_date(date):
    try:
        sec_df = get_all_securities(["etf"], date=date)
    except NameError:
        raise RuntimeError("get_all_securities is unavailable. Run this in JoinQuant or use cached PANEL_CSV.")
    except Exception as err:
        print("get_all_securities failed", date, err)
        return []
    if sec_df is None or sec_df.empty:
        return []
    out = []
    for code, row in sec_df.iterrows():
        try:
            start_date = row.get("start_date", None)
            if pd.isnull(start_date):
                start_date = get_security_info(code).start_date
            start_date = pd.Timestamp(start_date).date()
            feature_dt = pd.Timestamp(date).date()
            if feature_dt - start_date < datetime.timedelta(days=MIN_LISTING_DAYS):
                continue
            name = row.get("display_name", "")
            if should_exclude_name(name):
                continue
            out.append(code)
        except Exception:
            continue
    return out


def safe_ratio(a, b):
    if pd.isnull(a) or pd.isnull(b) or float(b) == 0.0:
        return np.nan
    return float(a) / float(b)


def safe_ret(close, days):
    if len(close) <= days:
        return np.nan
    base = close.iloc[-days - 1]
    if pd.isnull(base) or base <= 0:
        return np.nan
    return close.iloc[-1] / base - 1.0


def calc_trend_metrics(close, days):
    if len(close) <= days:
        return {
            "ann": np.nan,
            "r2": np.nan,
            "score": np.nan,
            "vol": np.nan,
            "score_vol_adj": np.nan,
            "simple_ann": np.nan,
        }
    recent = pd.Series(close.iloc[-(days + 1):].astype(float).values)
    if recent.isnull().any() or (recent <= 0).any():
        return {
            "ann": np.nan,
            "r2": np.nan,
            "score": np.nan,
            "vol": np.nan,
            "score_vol_adj": np.nan,
            "simple_ann": np.nan,
        }
    y = np.log(recent.values)
    x = np.arange(len(y))
    weights = np.linspace(1.0, TREND_WEIGHT_END, len(y))
    try:
        slope, intercept = np.polyfit(x, y, 1, w=weights)
    except Exception:
        return {
            "ann": np.nan,
            "r2": np.nan,
            "score": np.nan,
            "vol": np.nan,
            "score_vol_adj": np.nan,
            "simple_ann": np.nan,
        }
    ann = math.exp(slope * 250.0) - 1.0
    fit = slope * x + intercept
    ss_res = np.sum(weights * (y - fit) ** 2)
    ss_tot = np.sum(weights * (y - np.mean(y)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else 0.0
    r2 = max(0.0, min(1.0, float(r2)))
    daily_returns = recent.pct_change().dropna()
    vol = float(daily_returns.std() * math.sqrt(250.0)) if len(daily_returns) > 1 else np.nan
    period_ret = recent.iloc[-1] / recent.iloc[0] - 1.0
    simple_ann = (1.0 + period_ret) ** (250.0 / float(days)) - 1.0 if 1.0 + period_ret > 0 else np.nan
    score = ann * r2
    score_vol_adj = score * 0.20 / max(vol, 0.05) if not pd.isnull(vol) else np.nan
    return {
        "ann": ann,
        "r2": r2,
        "score": score,
        "vol": vol,
        "score_vol_adj": score_vol_adj,
        "simple_ann": simple_ann,
    }


def calc_one_etf_features(code, price_df):
    df = price_df.sort_values("time").copy()
    if len(df) < max(40, int(LOOKBACK_DAYS * 0.8)):
        return None
    for col in ["open", "high", "low", "close", "volume", "money"]:
        if col not in df.columns:
            return None
    close = pd.Series(df["close"].astype(float).values)
    high = pd.Series(df["high"].astype(float).values)
    low = pd.Series(df["low"].astype(float).values)
    volume = pd.Series(df["volume"].astype(float).values)
    money = pd.Series(df["money"].astype(float).values)
    if close.isnull().any() or close.iloc[-1] <= 0:
        return None

    ret = close.pct_change()
    rec = {"code": code}
    rec["ret_1"] = safe_ret(close, 1)
    rec["ret_5"] = safe_ret(close, 5)
    rec["ret_10"] = safe_ret(close, 10)
    rec["ret_20"] = safe_ret(close, 20)
    rec["ret_60"] = safe_ret(close, 60)
    rec["vol_5"] = ret.tail(5).std()
    rec["vol_20"] = ret.tail(20).std()
    rec["vol_60"] = ret.tail(60).std()
    rec["close_to_ma20"] = safe_ratio(close.iloc[-1], close.tail(20).mean()) - 1.0
    rec["close_to_ma60"] = safe_ratio(close.iloc[-1], close.tail(60).mean()) - 1.0
    rec["ma5_to_ma20"] = safe_ratio(close.tail(5).mean(), close.tail(20).mean()) - 1.0
    rec["ma20_to_ma60"] = safe_ratio(close.tail(20).mean(), close.tail(60).mean()) - 1.0
    rec["drawdown_20"] = safe_ratio(close.iloc[-1], close.tail(20).max()) - 1.0
    rec["drawdown_60"] = safe_ratio(close.iloc[-1], close.tail(60).max()) - 1.0
    rec["amp_20"] = (high.tail(20) / low.tail(20) - 1.0).replace([np.inf, -np.inf], np.nan).mean()
    rec["amp_60"] = (high.tail(60) / low.tail(60) - 1.0).replace([np.inf, -np.inf], np.nan).mean()
    rec["money_mean_20"] = money.tail(20).mean()
    rec["money_ratio_5_20"] = safe_ratio(money.tail(5).mean(), money.tail(20).mean()) - 1.0
    rec["money_ratio_20_60"] = safe_ratio(money.tail(20).mean(), money.tail(60).mean()) - 1.0
    rec["volume_ratio_5_20"] = safe_ratio(volume.tail(5).mean(), volume.tail(20).mean()) - 1.0
    rec["volume_ratio_20_60"] = safe_ratio(volume.tail(20).mean(), volume.tail(60).mean()) - 1.0
    rec["max_ret_20"] = ret.tail(20).max()
    rec["min_ret_20"] = ret.tail(20).min()
    for w in TREND_WINDOWS:
        tm = calc_trend_metrics(close, w)
        rec["trend_ann_%s" % w] = tm["ann"]
        rec["trend_r2_%s" % w] = tm["r2"]
        rec["trend_score_%s" % w] = tm["score"]
        rec["trend_vol_%s" % w] = tm["vol"]
        rec["trend_score_vol_adj_%s" % w] = tm["score_vol_adj"]
        rec["trend_simple_ann_%s" % w] = tm["simple_ann"]
    return rec


def fetch_feature_rows(feature_date, etfs):
    rows = []
    fields = ["open", "high", "low", "close", "volume", "money"]
    for etf_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        try:
            price_df = get_price(
                etf_chunk,
                end_date=feature_date,
                frequency="daily",
                fields=fields,
                count=PRICE_HISTORY_COUNT,
                panel=False,
                fq="pre",
                skip_paused=False,
            )
        except Exception as err:
            print("price feature chunk failed", feature_date, err)
            continue
        if price_df is None or price_df.empty:
            continue
        for code, one in price_df.groupby("code"):
            rec = calc_one_etf_features(code, one)
            if rec is not None:
                rows.append(rec)
    if len(rows) == 0:
        return pd.DataFrame()
    out = pd.DataFrame(rows)
    return out.replace([np.inf, -np.inf], np.nan)


def fetch_future_returns(feature_date, etfs):
    try:
        td = pd.to_datetime(get_trade_days(start_date=feature_date, count=LABEL_HORIZON_DAYS + 1))
    except Exception as err:
        print("future trade days failed", feature_date, err)
        return pd.DataFrame(columns=["code", "future_ret_5d", "next_date"])
    if len(td) < LABEL_HORIZON_DAYS + 1:
        return pd.DataFrame(columns=["code", "future_ret_5d", "next_date"])
    next_date = pd.Timestamp(td[-1]).normalize()
    rows = []
    for etf_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        try:
            px = get_price(
                etf_chunk,
                start_date=feature_date,
                end_date=next_date,
                frequency="daily",
                fields=["close"],
                panel=False,
                fq="pre",
                skip_paused=False,
            )
        except Exception as err:
            print("future price chunk failed", feature_date, err)
            continue
        if px is None or px.empty:
            continue
        for code, one in px.groupby("code"):
            one = one.sort_values("time")
            if len(one) < 2:
                continue
            start_close = float(one.iloc[0]["close"])
            end_close = float(one.iloc[-1]["close"])
            if start_close <= 0:
                continue
            rows.append({
                "code": code,
                "future_ret_5d": end_close / start_close - 1.0,
                "next_date": next_date,
            })
    return pd.DataFrame(rows)


def add_rank_features(df):
    out = df.copy()
    for col in RAW_FEATURE_COLS:
        if col in out.columns:
            out["rank_" + col] = out[col].rank(pct=True)
    return out


def add_volume_ab_feature(df):
    # The only V2.1 feature change: volume confirmation applied after V2 ranks.
    out = df.copy()
    out["ret_5_x_rank_money_ratio_5_20"] = out["ret_5"] * out["rank_money_ratio_5_20"]
    out["trend_score_20_x_rank_money_ratio_5_20"] = out["trend_score_20"] * out["rank_money_ratio_5_20"]
    return out.replace([np.inf, -np.inf], np.nan)


def add_pool_context_features(df):
    out = df.copy()
    ann_col = "trend_ann_%s" % POOL_CONTEXT_WINDOW
    vol_col = "trend_vol_%s" % POOL_CONTEXT_WINDOW
    if ann_col in out.columns and len(out) > 0:
        out["pool_breadth_%s" % POOL_CONTEXT_WINDOW] = float((out[ann_col] > 0).mean())
    else:
        out["pool_breadth_%s" % POOL_CONTEXT_WINDOW] = np.nan
    if vol_col in out.columns and len(out) > 0:
        out["pool_median_vol_%s" % POOL_CONTEXT_WINDOW] = float(out[vol_col].median())
    else:
        out["pool_median_vol_%s" % POOL_CONTEXT_WINDOW] = np.nan
    return out


def build_one_feature_date(feature_date):
    etfs = get_etf_universe_on_date(feature_date)
    if len(etfs) == 0:
        return pd.DataFrame()
    feature_df = fetch_feature_rows(feature_date, etfs)
    if feature_df.empty:
        return pd.DataFrame()
    if "money_mean_20" in feature_df.columns:
        feature_df = feature_df[feature_df["money_mean_20"] >= MIN_AVG_MONEY_20].copy()
    if feature_df.empty:
        return pd.DataFrame()
    feature_df = add_pool_context_features(feature_df)
    label_df = fetch_future_returns(feature_date, list(feature_df["code"]))
    if label_df.empty:
        return pd.DataFrame()
    out = feature_df.merge(label_df, on="code", how="inner")
    if out.empty:
        return out
    out = add_rank_features(out)
    out = add_volume_ab_feature(out)
    out["feature_date"] = pd.Timestamp(feature_date).normalize()
    out["rebalance_date"] = out["feature_date"]
    out["target_alpha_5d"] = out["future_ret_5d"] - out["future_ret_5d"].median()
    return out

In [ ]:
# =========================
# 3. Build or load weekly ETF panel
# =========================
def build_weekly_panel():
    feature_dates = get_weekly_feature_dates(START_DATE, END_DATE)
    print("feature weeks:", len(feature_dates), "from", feature_dates[0] if feature_dates else None, "to", feature_dates[-1] if feature_dates else None)
    parts = []
    for dt in tqdm(feature_dates, desc="build etf weekly panel"):
        one = build_one_feature_date(dt)
        if one is not None and not one.empty:
            parts.append(one)
        if len(parts) % 20 == 0:
            gc.collect()
    if len(parts) == 0:
        raise RuntimeError("No ETF weekly samples were built. Check ETF universe/data access.")
    panel = pd.concat(parts, ignore_index=True, sort=False)
    # JoinQuant sometimes returns stale/placeholder future prices near the data boundary.
    # If almost every ETF has exactly zero future return on one feature_date, treat that
    # date as an incomplete-label date and drop it from training/evaluation.
    label_quality = panel.groupby("feature_date")["future_ret_5d"].agg(["count", lambda s: (s == 0).mean()])
    label_quality.columns = ["sample_count", "zero_return_rate"]
    bad_dates = set(label_quality[label_quality["zero_return_rate"] >= 0.98].index)
    if bad_dates:
        print("drop incomplete label dates:", sorted([str(pd.Timestamp(x).date()) for x in bad_dates]))
        panel = panel[~panel["feature_date"].isin(bad_dates)].copy()
    date_cols = ["feature_date", "rebalance_date", "next_date"]
    for c in date_cols:
        if c in panel.columns:
            panel[c] = pd.to_datetime(panel[c])
    return panel


if REBUILD_DATA or (not os.path.exists(PANEL_CSV)):
    panel_df = build_weekly_panel()
    panel_df.to_csv(PANEL_CSV, index=False)
else:
    panel_df = pd.read_csv(PANEL_CSV)
    for c in ["feature_date", "rebalance_date", "next_date"]:
        if c in panel_df.columns:
            panel_df[c] = pd.to_datetime(panel_df[c])

print("panel shape:", panel_df.shape)
print(panel_df[["feature_date", "next_date"]].agg(["min", "max"]))
print("sample per week:", panel_df.groupby("feature_date")["code"].count().describe())
display(panel_df.head())

build etf weekly panel:   0%|          | 0/282 [00:00<?, ?it/s]

feature weeks: 282 from 2021-01-08 00:00:00 to 2026-06-30 00:00:00


build etf weekly panel:  50%|█████     | 142/282 [17:34<23:58, 10.27s/it]

In [ ]:
# =========================
# 4. Train, score, and export model bundles
# =========================
META_COLS = ["code", "feature_date", "next_date", "future_ret_5d", "target_alpha_5d"]
SCORE_EXPORT_COLS = META_COLS + ["score", "model_name", "model_file", "target_name", "train_tag"]


def clean_train_df(df, target_col):
    cols = META_COLS + FEATURE_COLS
    cols = [c for c in cols if c in df.columns]
    out = df.loc[:, cols].replace([np.inf, -np.inf], np.nan)
    out = out[~out[target_col].isnull()].copy()
    return out


def train_lgb_or_fallback(train_df, target_col, feature_cols):
    X_raw = train_df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
    fill_values = X_raw.median().to_dict()
    X = X_raw.fillna(pd.Series(fill_values)).fillna(0)
    y = train_df[target_col].astype(float).values
    try:
        import lightgbm as lgb
        dtrain = lgb.Dataset(X[feature_cols], label=y, feature_name=list(feature_cols), free_raw_data=True)
        model = lgb.train(LGB_PARAMS, dtrain, num_boost_round=NUM_BOOST_ROUND)
        backend = "lightgbm"
        del dtrain
    except Exception as err:
        print("LightGBM unavailable or failed, fallback to sklearn RandomForestRegressor:", err)
        from sklearn.ensemble import RandomForestRegressor
        model = RandomForestRegressor(
            n_estimators=260,
            max_depth=6,
            min_samples_leaf=30,
            random_state=RANDOM_SEED,
            n_jobs=1,
        )
        model.fit(X[feature_cols], y)
        backend = "sklearn_random_forest"
    del X_raw, X, y
    gc.collect()
    return model, fill_values, backend


def predict_model(model, fill_values, df, feature_cols):
    X_raw = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
    X = X_raw.fillna(pd.Series(fill_values)).fillna(0)
    pred = np.asarray(model.predict(X[feature_cols])).reshape(-1).astype(float)
    del X_raw, X
    gc.collect()
    return pred


def make_bundle(model, fill_values, backend, target_name, target_col, train_tag, train_start, train_end):
    return {
        "objective": "etf_ml_weekly_lgb_v2_1",
        "model": model,
        "model_backend": backend,
        "target_name": target_name,
        "target_col": target_col,
        "feature_cols": list(FEATURE_COLS),
        "raw_feature_cols": list(RAW_FEATURE_COLS),
        "rank_feature_cols": list(RANK_FEATURE_COLS),
        "context_feature_cols": list(CONTEXT_FEATURE_COLS),
        "trend_windows": list(TREND_WINDOWS),
        "trend_weight_end": TREND_WEIGHT_END,
        "price_history_count": PRICE_HISTORY_COUNT,
        "pool_context_window": POOL_CONTEXT_WINDOW,
        "fill_values": dict(fill_values),
        "lookback_days": LOOKBACK_DAYS,
        "label_horizon_days": LABEL_HORIZON_DAYS,
        "min_listing_days": MIN_LISTING_DAYS,
        "min_avg_money_20": MIN_AVG_MONEY_20,
        "run_tag": RUN_TAG,
        "volume_ab_mode": VOLUME_AB_MODE,
        "exclude_name_keywords": list(EXCLUDE_NAME_KEYWORDS),
        "stock_num": STOCK_NUM,
        "benchmark": BENCHMARK,
        "train_tag": train_tag,
        "train_start": train_start,
        "train_end": train_end,
        "research_version": "etf_ml_v2_1_v2_baseline_volume_ab",
        "created_note": "V2 data/label/train path; one optional volume-confirmation interaction after V2 rank features",
    }


def append_csv(df, path):
    if df is None or df.empty:
        return
    write_header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=write_header, index=False)


# 重新运行本 cell 时避免把旧 score 追加进去。
if os.path.exists(SCORE_CSV):
    os.remove(SCORE_CSV)
if os.path.exists(MODEL_MANIFEST_CSV):
    os.remove(MODEL_MANIFEST_CSV)

score_keep_parts = []
manifest_rows = []
panel_df = clean_train_df(panel_df, "target_alpha_5d")

for train_tag, train_start, train_end in tqdm(TRAIN_WINDOWS, desc="train windows"):
    train_start_ts = pd.Timestamp(train_start)
    train_end_ts = pd.Timestamp(train_end)
    # A model trained "through train_end" may only use samples whose label is already known
    # by train_end. The first live rebalance after train_end will usually score the previous
    # trading day's features, so OOS score/evaluation is defined by next_date > train_end.
    train_mask = (panel_df["feature_date"] >= train_start_ts) & (panel_df["next_date"] <= train_end_ts)
    score_mask = panel_df["next_date"] > train_end_ts
    if not bool(train_mask.any()) or not bool(score_mask.any()):
        print("skip window", train_tag, "train", int(train_mask.sum()), "score", int(score_mask.sum()))
        continue

    # Keep one feature-bearing OOS table per train window, not one per model.
    score_base = panel_df.loc[score_mask, META_COLS + FEATURE_COLS].copy()

    for target_name, target_col in tqdm(TARGET_SPECS, desc=train_tag, leave=False):
        train_df = clean_train_df(panel_df.loc[train_mask], target_col)
        if train_df.empty:
            print("skip target", target_col, "empty train")
            continue
        print("training", train_tag, target_col, "samples", train_df.shape[0], "weeks", train_df["feature_date"].nunique())
        model, fill_values, backend = train_lgb_or_fallback(train_df, target_col, FEATURE_COLS)
        bundle = make_bundle(model, fill_values, backend, target_name, target_col, train_tag, train_start, train_end)
        model_file = "model_etf_ml_v2_1_lgb_%s_%s_%s.pkl" % (target_name, train_tag, RUN_TAG)
        model_path = os.path.join(OUT_DIR, model_file)
        with open(model_path, "wb") as f:
            pickle.dump(bundle, f, protocol=2)

        scored = score_base.loc[:, META_COLS].copy()
        scored["score"] = predict_model(model, fill_values, score_base, FEATURE_COLS)
        scored["model_name"] = "etf_ml_v2_1_lgb_%s_%s_%s" % (target_name, train_tag, RUN_TAG)
        scored["model_file"] = model_file
        scored["target_name"] = target_name
        scored["train_tag"] = train_tag
        scored = scored.loc[:, SCORE_EXPORT_COLS]
        append_csv(scored, SCORE_CSV)
        score_keep_parts.append(scored)

        manifest_rows.append({
            "model_name": scored["model_name"].iloc[0],
            "model_file": model_file,
            "model_backend": backend,
            "target_col": target_col,
            "train_start": train_start,
            "train_end": train_end,
            "train_samples": len(train_df),
            "train_weeks": train_df["feature_date"].nunique(),
            "feature_count": len(FEATURE_COLS),
        })
        del scored, model, bundle, train_df, fill_values
        gc.collect()

    del score_base
    gc.collect()

score_df = pd.concat(score_keep_parts, ignore_index=True, sort=False) if score_keep_parts else pd.DataFrame(columns=SCORE_EXPORT_COLS)
manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MODEL_MANIFEST_CSV, index=False)
print("score shape:", score_df.shape)
print("score columns:", list(score_df.columns))
display(manifest_df)



In [ ]:
# =========================
# 5. Offline top3 proxy evaluation
# =========================
def summarize_returns(ret_series):
    r = pd.Series(ret_series).dropna().astype(float)
    if len(r) == 0:
        return {"weeks": 0}
    cum_ret = float((1.0 + r).prod() - 1.0)
    mean_ret = float(r.mean())
    std_ret = float(r.std())
    sharpe_weekly = np.nan if std_ret <= 0 or pd.isnull(std_ret) else mean_ret / std_ret
    nav = (1.0 + r).cumprod()
    dd = nav / nav.cummax() - 1.0
    return {
        "weeks": int(len(r)),
        "cum_ret": cum_ret,
        "mean_weekly_ret": mean_ret,
        "win_rate": float((r > 0).mean()),
        "weekly_sharpe": float(sharpe_weekly) if not pd.isnull(sharpe_weekly) else np.nan,
        "max_drawdown": float(dd.min()),
    }


def build_weekly_proxy(score_data):
    rows = []
    rng = np.random.RandomState(RANDOM_SEED)
    for (model_name, dt), gdf in tqdm(score_data.groupby(["model_name", "feature_date"]), desc="weekly proxy"):
        gdf = gdf.sort_values("score", ascending=False).copy()
        top = gdf.head(min(STOCK_NUM, len(gdf)))
        if top.empty:
            continue
        median_ret = float(gdf["future_ret_5d"].median())
        random_count = min(STOCK_NUM, len(gdf))
        random_ret = float(gdf.sample(random_count, random_state=int(rng.randint(0, 1000000)))["future_ret_5d"].mean())
        rows.append({
            "model_name": model_name,
            "model_file": top["model_file"].iloc[0],
            "target_name": top["target_name"].iloc[0],
            "train_tag": top["train_tag"].iloc[0],
            "feature_date": dt,
            "next_date": top["next_date"].iloc[0],
            "target_count": len(top),
            "targets": ",".join(top["code"].tolist()),
            "top3_ret_5d": float(top["future_ret_5d"].mean()),
            "universe_median_ret_5d": median_ret,
            "excess_vs_median_5d": float(top["future_ret_5d"].mean() - median_ret),
            "random_top3_ret_5d": random_ret,
            "excess_vs_random_5d": float(top["future_ret_5d"].mean() - random_ret),
        })
    return pd.DataFrame(rows)


weekly_df = build_weekly_proxy(score_df) if not score_df.empty else pd.DataFrame()
summary_rows = []
if not weekly_df.empty:
    for model_name, gdf in weekly_df.groupby("model_name"):
        ret_stats = summarize_returns(gdf["top3_ret_5d"])
        excess_median_stats = summarize_returns(gdf["excess_vs_median_5d"])
        excess_random_stats = summarize_returns(gdf["excess_vs_random_5d"])
        row = {"model_name": model_name}
        row.update({"proxy_" + k: v for k, v in ret_stats.items()})
        row.update({"excess_median_" + k: v for k, v in excess_median_stats.items()})
        row.update({"excess_random_" + k: v for k, v in excess_random_stats.items()})
        meta = gdf.iloc[0][["model_file", "target_name", "train_tag"]].to_dict()
        row.update(meta)
        summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    summary_df = summary_df.sort_values(["proxy_cum_ret", "excess_median_cum_ret"], ascending=[False, False])

weekly_df.to_csv(WEEKLY_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)
display(summary_df)
display(weekly_df.tail(10))

In [ ]:
# =========================
# 6. Latest targets for quick inspection
# =========================
latest_rows = []
if not score_df.empty:
    for model_name, gdf in score_df.groupby("model_name"):
        latest_date = gdf["feature_date"].max()
        one = gdf[gdf["feature_date"] == latest_date].sort_values("score", ascending=False).head(STOCK_NUM).copy()
        for rank_idx, (_, row) in enumerate(one.iterrows(), 1):
            latest_rows.append({
                "model_name": model_name,
                "model_file": row["model_file"],
                "feature_date": latest_date,
                "rank": rank_idx,
                "code": row["code"],
                "score": row["score"],
                "future_ret_5d": row.get("future_ret_5d", np.nan),
            })
latest_targets_df = pd.DataFrame(latest_rows)
latest_targets_df.to_csv(LATEST_TARGETS_CSV, index=False)
display(latest_targets_df.head(30))
print("outputs saved:")
for p in [PANEL_CSV, SCORE_CSV, SUMMARY_CSV, WEEKLY_CSV, LATEST_TARGETS_CSV, MODEL_MANIFEST_CSV]:
    print(" -", p)

## Self Review

交付前自查：

- 特征只用 `feature_date` 及以前历史日线数据；60 日收益/趋势使用 61 个价格点，避免天然缺失。
- Label 使用 `feature_date` 之后 5 个交易日收益，只用于训练和离线评估，不进入回测文件。
- 训练和评分按可观测 label 切分：`next_date <= train_end` 训练，`next_date > train_end` 评分，避免边界周 label 穿越。
- 无 early stopping，避免小 valid 窗口扰动模型。
- Notebook 和回测文件使用同一套 `RAW_FEATURE_COLS` / `rank_*` / `CONTEXT_FEATURE_COLS` / 趋势 R² 公式 / 流动性过滤 / 名称过滤。
- 回测文件只加载导出的 pkl，并在每周调仓时用 `context.previous_date` 重算 feature。

重要限制：离线 proxy 使用 feature_date close 到未来 5 日 close 的收益评估，不能完全替代 JoinQuant 真实下单回测；最终仍以回测文件加载 pkl 的结果为准。

In [ ]:
# =========================
# 7. Default second liquidity run: money5000w
# =========================
# Cells 3-6 above execute the first profile (money2000w). This cell uses the
# same V2 helpers and training functions to export the remaining configured profile.
def configure_liquidity_profile(liquidity_tag):
    global LIQUIDITY_TAG, RUN_TAG, MIN_AVG_MONEY_20
    global OUT_DIR, PANEL_CSV, SCORE_CSV, SUMMARY_CSV, WEEKLY_CSV
    global LATEST_TARGETS_CSV, MODEL_MANIFEST_CSV
    if liquidity_tag not in LIQUIDITY_THRESHOLDS:
        raise ValueError("invalid liquidity tag: %s" % liquidity_tag)
    LIQUIDITY_TAG = liquidity_tag
    RUN_TAG = LIQUIDITY_TAG + "_" + VOLUME_AB_MODE
    MIN_AVG_MONEY_20 = LIQUIDITY_THRESHOLDS[LIQUIDITY_TAG]
    OUT_DIR = "etf_ml_v2_1_" + RUN_TAG + "_outputs"
    PANEL_CSV = os.path.join(OUT_DIR, "etf_ml_v2_weekly_panel.csv")
    SCORE_CSV = os.path.join(OUT_DIR, "etf_ml_v2_score_panel.csv")
    SUMMARY_CSV = os.path.join(OUT_DIR, "etf_ml_v2_model_summary.csv")
    WEEKLY_CSV = os.path.join(OUT_DIR, "etf_ml_v2_weekly_top3_proxy.csv")
    LATEST_TARGETS_CSV = os.path.join(OUT_DIR, "etf_ml_v2_latest_targets.csv")
    MODEL_MANIFEST_CSV = os.path.join(OUT_DIR, "etf_ml_v2_model_manifest.csv")
    if not os.path.exists(OUT_DIR):
        os.makedirs(OUT_DIR)


def build_load_train_evaluate_profile(liquidity_tag):
    configure_liquidity_profile(liquidity_tag)
    print("\n===== V2.1 profile %s: min_avg_money_20=%.0f =====" % (LIQUIDITY_TAG, MIN_AVG_MONEY_20))
    if REBUILD_DATA or (not os.path.exists(PANEL_CSV)):
        profile_panel = build_weekly_panel()
        profile_panel.to_csv(PANEL_CSV, index=False)
    else:
        profile_panel = pd.read_csv(PANEL_CSV)
        for c in ["feature_date", "rebalance_date", "next_date"]:
            if c in profile_panel.columns:
                profile_panel[c] = pd.to_datetime(profile_panel[c])
    print("panel shape:", profile_panel.shape)
    profile_panel = clean_train_df(profile_panel, "target_alpha_5d")

    if os.path.exists(SCORE_CSV):
        os.remove(SCORE_CSV)
    if os.path.exists(MODEL_MANIFEST_CSV):
        os.remove(MODEL_MANIFEST_CSV)
    score_parts = []
    manifest_rows = []
    for train_tag, train_start, train_end in tqdm(TRAIN_WINDOWS, desc="train %s" % LIQUIDITY_TAG):
        train_start_ts = pd.Timestamp(train_start)
        train_end_ts = pd.Timestamp(train_end)
        train_mask = (profile_panel["feature_date"] >= train_start_ts) & (profile_panel["next_date"] <= train_end_ts)
        score_mask = profile_panel["next_date"] > train_end_ts
        if not bool(train_mask.any()) or not bool(score_mask.any()):
            print("skip window", train_tag, "train", int(train_mask.sum()), "score", int(score_mask.sum()))
            continue
        score_base = profile_panel.loc[score_mask, META_COLS + FEATURE_COLS].copy()
        for target_name, target_col in tqdm(TARGET_SPECS, desc=train_tag, leave=False):
            train_df = clean_train_df(profile_panel.loc[train_mask], target_col)
            if train_df.empty:
                print("skip target", target_col, "empty train")
                continue
            print("training", train_tag, target_col, "samples", train_df.shape[0], "weeks", train_df["feature_date"].nunique())
            model, fill_values, backend = train_lgb_or_fallback(train_df, target_col, FEATURE_COLS)
            bundle = make_bundle(model, fill_values, backend, target_name, target_col, train_tag, train_start, train_end)
            model_file = "model_etf_ml_v2_1_lgb_%s_%s_%s.pkl" % (target_name, train_tag, RUN_TAG)
            with open(os.path.join(OUT_DIR, model_file), "wb") as f:
                pickle.dump(bundle, f, protocol=2)
            scored = profile_panel.loc[score_mask, META_COLS].copy()
            scored["score"] = predict_model(model, fill_values, score_base, FEATURE_COLS)
            scored["model_name"] = "etf_ml_v2_1_lgb_%s_%s_%s" % (target_name, train_tag, RUN_TAG)
            scored["model_file"] = model_file
            scored["target_name"] = target_name
            scored["train_tag"] = train_tag
            scored = scored.loc[:, SCORE_EXPORT_COLS]
            append_csv(scored, SCORE_CSV)
            score_parts.append(scored)
            manifest_rows.append({
                "model_name": scored["model_name"].iloc[0], "model_file": model_file,
                "model_backend": backend, "target_col": target_col,
                "train_start": train_start, "train_end": train_end,
                "train_samples": len(train_df), "train_weeks": train_df["feature_date"].nunique(),
                "feature_count": len(FEATURE_COLS),
            })
            del scored, model, bundle, train_df, fill_values
            gc.collect()
        del score_base
        gc.collect()

    profile_score = pd.concat(score_parts, ignore_index=True, sort=False) if score_parts else pd.DataFrame(columns=SCORE_EXPORT_COLS)
    pd.DataFrame(manifest_rows).to_csv(MODEL_MANIFEST_CSV, index=False)
    profile_weekly = build_weekly_proxy(profile_score) if not profile_score.empty else pd.DataFrame()
    summary_rows = []
    if not profile_weekly.empty:
        for model_name, gdf in profile_weekly.groupby("model_name"):
            row = {"model_name": model_name}
            row.update({"proxy_" + k: v for k, v in summarize_returns(gdf["top3_ret_5d"]).items()})
            row.update({"excess_median_" + k: v for k, v in summarize_returns(gdf["excess_vs_median_5d"]).items()})
            row.update({"excess_random_" + k: v for k, v in summarize_returns(gdf["excess_vs_random_5d"]).items()})
            row.update(gdf.iloc[0][["model_file", "target_name", "train_tag"]].to_dict())
            summary_rows.append(row)
    profile_summary = pd.DataFrame(summary_rows)
    if not profile_summary.empty:
        profile_summary = profile_summary.sort_values(["proxy_cum_ret", "excess_median_cum_ret"], ascending=[False, False])
    profile_weekly.to_csv(WEEKLY_CSV, index=False)
    profile_summary.to_csv(SUMMARY_CSV, index=False)

    latest_rows = []
    for model_name, gdf in profile_score.groupby("model_name"):
        latest_date = gdf["feature_date"].max()
        one = gdf[gdf["feature_date"] == latest_date].sort_values("score", ascending=False).head(STOCK_NUM)
        for rank_idx, (_, row) in enumerate(one.iterrows(), 1):
            latest_rows.append({"model_name": model_name, "model_file": row["model_file"],
                                "feature_date": latest_date, "rank": rank_idx, "code": row["code"],
                                "score": row["score"], "future_ret_5d": row.get("future_ret_5d", np.nan)})
    pd.DataFrame(latest_rows).to_csv(LATEST_TARGETS_CSV, index=False)
    display(profile_summary)
    print("outputs saved for", LIQUIDITY_TAG, OUT_DIR)
    del profile_panel, profile_score, profile_weekly, profile_summary
    gc.collect()


# First profile was produced by cells 3-6. Produce every remaining profile by default.
for _liquidity_tag in LIQUIDITY_TAGS_TO_RUN[1:]:
    build_load_train_evaluate_profile(_liquidity_tag)
